# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:

- Santiago Lastra Ramírez
- Raul Aldair Ureña Moreno

Url: [https://github.com/slastrar/Proyecto-AlgoritmosOptimizacion](https://github.com/slastrar/Proyecto-AlgoritmosOptimizacion)<br>

Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible.

Los datos son:
> Número de actores: 10<br>
> Número de tomas : 30






                                        

In [1]:
import time
import math
import numpy as np

## 1.

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>

Sin restricciones, cada una de las 30 tomas podría en el peor de los casos requerir su propio día de grabación, por lo que se consideran hasta 30 días posibles; cada toma elige libremente entre ellos, dando

$$\boxed{30^{30} \approx 2,06 \times 10^{44} \text{ posibilidades}}$$




¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones?

Algo importante a tener en cuenta es que si son 30 tomas y solo se pueden como máximo 6 tomas al día entonces el mínimo de días son 30/6 = 5. Usar más días podría reducir el coste en algunos casos (por ejemplo, con 6 tomas, máximo 3 por día y 3 actores que aparecen en 2 tomas cada uno, con 2 días el coste mínimo es 4 y con 3 días es 3), pero en este trabajo se fija el modelo en 5 días de 6 tomas cada uno. Con esto en mente entonces el objetivo es repartir las 30 tomas en 5 grupos de 6 tomas (para cada día) que minimice la cantidad de actores por día.

Entonces para calcular la cantidad de posibilidades el objetivo es hacer para cada día un cálculo de una combinatoria sin repetición, o sea:

$$C_{n,k}=\binom{n}{k}=\frac{n!}{k!(n-k)!}$$

Para el primer día $n=30$ y $k=6$
$$C_{\text{1 dia}}=\binom{30}{6}=\frac{30!}{6! \cdot 24!} = 593775$$

Para el segundo día como ya asignamos 6, entonces $n=24$ y $k=6$
$$C_{\text{2 dia}}=\binom{24}{6}=\frac{24!}{6! \cdot 18!} = 134596$$

Para el tercer día $n=18$ y $k=6$, para el cuarto $n=12$ y $k=6$ y para el quinto $n=6$ y $k=6$

$$C_{\text{3 dia}}=\binom{18}{6}=\frac{18!}{6! \cdot 12!} = 18564$$
$$C_{\text{4 dia}}=\binom{12}{6}=\frac{12!}{6! \cdot 6!} = 924$$
$$C_{\text{5 dia}}=\binom{6}{6}=\frac{6!}{6! \cdot 0!} = 1$$

Y como cada elección depende de la anterior entonces se multiplican los valores, obteniendo así que la cantidad de soluciones es de $$\boxed{1.37 \times 10^{18} \text{ posibilidades}}$$

Pero los días no tienen un orden: da igual que un grupo de 6 tomas se grabe el día 1 o el día 5, el costo es el mismo. Entonces cada planificación está contada $5! = 120$ veces y el número de planificaciones realmente distintas es

$$\boxed{\frac{30!}{(6!)^5 \cdot 5!} \approx 1.14 \times 10^{16} \text{ posibilidades}}$$

Si no se fijara el número de días (pudiendo ir de 5 hasta 30, con máximo 6 tomas por día y días sin orden), el total de planificaciones sería de aproximadamente $7.26 \times 10^{23}$. 

## 2.

Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)


La estructura de datos que utilizaré para este trabajo va a ser la misma que se plasma en el archivo de los datos, es decir, una matriz de $30 \times 10$ donde cada fila representa una toma y las $10$ entradas de esa fila nos dicen si el actor correspondiente participa o no en ella, representado con $1$ y $0$ respectivamente. Se elige esta estructura porque al ser una lista de listas, se puede acceder directamente a `matriz[toma][actor]` sin transformaciones previas, y para calcular el coste de un día basta con recorrer las filas de las tomas de ese día y revisar, columna por columna, si algún actor aparece en alguna de ellas.

La data queda así:

In [2]:
matriz = [
    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0], 
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 0, 1, 0, 0, 0],
    [1, 1, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 1, 0, 1, 0, 0, 0, 1, 0, 0],
    [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
    [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    [1, 1, 1, 0, 1, 0, 0, 1, 0, 0],
    [1, 1, 1, 1, 0, 1, 0, 0, 0, 0],
    [1, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
]

Además, al final se obtendrá un vector llamado `asignacion` de tamaño $30$ donde cada entrada pertenecerá a cada una de las tomas y el valor contenido en esa entrada dirá para qué día está asignada.

También se obtendrá un valor del 'coste' que hace referencia a la suma de cantidad de días que tuvo que asistir cada actor. Esto se explica formalmente en el siguiente punto.

## 3.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?


(*)¿Es un problema de maximización o minimización?

Lo que se busca es minimizar el número total de desplazamientos de actores, o sea, la suma de cuántos actores distintos tienen que acudir al estudio en cada uno de los 5 días de grabación.

Formalmente, se define el vector asignación $a=(a_1,\ldots,a_{30})$ el vector que representa la planificación de las tomas, donde $a_t \in \{1,2,3,4,5\}$ indica el día en que se realizará la toma $t$.

Para saber qué actores deben asistir cada día, se define la variable:

$$
y_{d,j} =
\begin{cases}
1, & \text{si el actor } j \text{ participa en al menos una toma del día } d,\\
0, & \text{en caso contrario.}
\end{cases}
$$

Es decir, si un actor aparece en varias tomas el mismo día, solo se cuenta una vez, ya que únicamente necesita desplazarse una vez a la grabación.

Con esta definición, la **función objetivo** (que es el número total de desplazamientos de actores) se calcula como:

$$
f(a)=\sum_{d=1}^{5}\sum_{j=1}^{10} y_{d,j}
$$

que simplemente suma la cantidad de actores que deben asistir en cada uno de los cinco días de grabación. El objetivo es minimizar este valor para reducir el costo total de los desplazamientos.

La planificación debe cumplir además la restricción de que cada uno de los cinco días tenga exactamente seis tomas asignadas:

$$
\sum_{t=1}^{30}[a_t=d]=6,
\qquad
\forall d\in\{1,2,3,4,5\}
$$


En resumen, el problema consiste en distribuir las 30 tomas entre los 5 días (6 tomas por día) tratando de que los actores que participan en varias escenas coincidan el mismo día. Así se reduce el número de días en los que cada actor debe desplazarse y, por lo tanto, el costo total de la producción.

## 4.

Diseña un algoritmo para resolver el problema por fuerza bruta

In [3]:
# Lo pongo entre comillas porque lo mejor es no ejecutarlo ya que tardaría mucho tiempo

'''
import itertools

# Devuelve el coste de esa solución
def funcion_objetivo(solucion,matriz):
    total = 0

    for dia in range(1, 6):
      actores = set()

      for toma in range(30):
        if solucion[toma] == dia:  # Se validan las tomas que están en el día
          for actor in range(10):
            if matriz[toma][actor] == 1:  # Se validan los actores que están en la toma
              actores.add(actor)

      total += len(actores)

    return total

##########################################
############## FUERZA BRUTA ##############
##########################################

# Asigna 6 tomas al día indicado y sigue con el resto de días
def repartir(dia, libres, solucion, matriz, mejor):
    if dia == 6:  # Los 5 días ya están completos
        costo = funcion_objetivo(solucion, matriz)

        if costo < mejor[0]:
            mejor[0] = costo
            mejor[1] = solucion.copy()
        return

    # Cada grupo de 6 tomas de las que quedan libres es una opción para este día
    for grupo in itertools.combinations(libres, 6):
        for toma in grupo:
            solucion[toma] = dia

        restantes = [toma for toma in libres if toma not in grupo]
        repartir(dia + 1, restantes, solucion, matriz, mejor)


mejor = [float('inf'), []]  # [mejor costo, mejor solución]

repartir(1, list(range(30)), [0]*30, matriz, mejor)

costo, mejor_solucion = mejor[0], mejor[1]
print("Costo minimo:", costo)

'''

'\nimport itertools\n\n# Devuelve el coste de esa solución\ndef funcion_objetivo(solucion,matriz):\n    total = 0\n\n    for dia in range(1, 6):\n      actores = set()\n\n      for toma in range(30):\n        if solucion[toma] == dia:  # Se validan las tomas que están en el día\n          for actor in range(10):\n            if matriz[toma][actor] == 1:  # Se validan los actores que están en la toma\n              actores.add(actor)\n\n      total += len(actores)\n\n    return total\n\n##########################################\n############## FUERZA BRUTA ##############\n##########################################\n\n# Asigna 6 tomas al día indicado y sigue con el resto de días\ndef repartir(dia, libres, solucion, matriz, mejor):\n    if dia == 6:  # Los 5 días ya están completos\n        costo = funcion_objetivo(solucion, matriz)\n\n        if costo < mejor[0]:\n            mejor[0] = costo\n            mejor[1] = solucion.copy()\n        return\n\n    # Cada grupo de 6 tomas de las q

## 5.

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

Para analizar el algoritmo se define $n$ como el número de tomas, $a$ como el número de actores y $k$ como el máximo de tomas por día, de modo que el número de días es $n/k$. En este problema $n=30$, $a=10$ y $k=6$, por lo que hay 5 días.

El algoritmo reparte las tomas día por día: a cada día se le elige un grupo de $k$ tomas entre las que todavía no tienen día y se pasa al siguiente día con las tomas que quedan. Cuando los 5 días están completos se calcula el costo con `funcion_objetivo` y se compara con el mejor encontrado. Como el grupo de cada día se elige una sola vez, cada planificación se recorre una única vez, por lo que el número de evaluaciones es el calculado en el punto 1:

$$N=\frac{n!}{(k!)^{n/k}} \approx 1.37 \times 10^{18}$$

La función `funcion_objetivo` recorre los $n/k$ días y, para cada uno, revisa las $n$ tomas y, de las que se graban ese día, recorre los $a$ actores. Con estos tres bucles anidados el número de operaciones es como máximo

$$\frac{n}{k} \cdot n \cdot a = n^2 \cdot \frac{a}{k} \quad \Rightarrow \quad O\left(n^2 \cdot \frac{a}{k}\right)$$

que para el problema son $30^2 \cdot 10/6 = 1500$ operaciones por evaluación. El resto del trabajo de cada llamada (marcar las tomas del grupo y construir la lista de las que quedan libres) es menor que la evaluación, así que no cambia el orden del costo.

Multiplicando el número de soluciones por el costo de evaluar cada una, el costo total de la fuerza bruta es

$$T_{FB}(n,a,k)=O\left(N \cdot n^2 \cdot \frac{a}{k}\right)=O\left(\frac{n!}{(k!)^{n/k}} \cdot n^2 \cdot \frac{a}{k}\right)$$

que con los números del problema equivale a $1.37 \times 10^{18} \cdot 1500 \approx 2.06 \times 10^{21}$ operaciones. Suponiendo que mi PC hace $10^7$ operaciones por segundo, entonces el algoritmo tardaría unos $2.06 \times 10^{14}$ segundos, es decir, alrededor de 6.5 millones de años, por lo que la fuerza bruta no es viable para este problema.

In [4]:
from math import factorial

n, a, k = 30, 10, 6
T = factorial(n) * n**2 * a / factorial(k)**(n / k)  / k     
ops_por_segundo = 10**7                          
segundos_por_ano = 3600 * 24 * 365

ops_codigo = T / ops_por_segundo

print(f"La cantidad de operaciones son: {T:.3e} operaciones -> {ops_codigo / segundos_por_ano:.2e} años")

La cantidad de operaciones son: 2.056e+21 operaciones -> 6.52e+06 años


## 6.

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

### Ramificación y Poda

Se guardaran los datos en un array de 30 posiciones, cada una de esas posiciones significa cada una de las tomas. Por ejemplo si el array resultante es `s = (1, 1, 2,...)` significa que las tomas 0 y 1 se graban el día 1 y la toma 2 el día 2. Como dije únicamente haremos uso de 5 días y su costo es el número total de actores que van cada día


El código se crea con 3 funciones antes de la principal:

1. `funcion_objetivo`: calcula el costo de una solución parcial, es decir, la suma de los actores distintos que hay cada día contando solo las tomas ya asignadas. 

2. `cota_inferior`: calcula el mínimo que costaría cualquier solución completa que parta de la solución parcial. Se hace actor por actor. Para cada actor se suman dos cosas:
   - los días a los que ya tiene que ir, por las tomas que ya tienen día;
   - los días nuevos que necesitaría como mínimo para sus tomas restantes. Sus tomas restantes caben primero en los huecos libres de los días a los que ya va y las que no caben obligan a abrir días nuevos, a razón de 6 tomas por día como máximo. Es decir, `nuevos = ⌈(restantes − huecos) / 6⌉`.

   Con los datos de este problema, los actores aparecen en 22, 14, 13, 15, 11, 8, 3, 4, 2 y 2 tomas. Antes de asignar ninguna toma, la cota es $4+3+3+3+2+2+1+1+1+1 = 21$, así que ninguna planificación puede costar menos de 21. Cuando se asignan tomas la cota sube, porque los huecos se van llenando. Por ejemplo, si las tomas 0 a 5 van todas el día 1, el costo parcial ya es 7 y la cota pasa a 25: el día 1 está lleno (0 huecos), así que todas las tomas restantes de cada actor obligan a días nuevos (el actor 1, con 19 tomas restantes, necesita al menos 4 días más, el actor 2 con 9 necesita 2, y así hasta sumar 18 días nuevos).

3. `crear_hijos`: crea las ramas de un nodo asignando la siguiente toma a cada día que todavía tenga hueco (máximo 6 tomas por día). Como los días son intercambiables, solo permite abrir el siguiente día sin usar. Por eso de `()` solo sale `(1,)`, de `(1, 1, 2)` salen `(1, 1, 2, 1)`, `(1, 1, 2, 2)` y `(1, 1, 2, 3)`, y cuando el día 1 ya tiene sus 6 tomas solo se puede abrir el día 2. Esto evita repetir la misma planificación con los días numerados distinto.

La función principal, `ramificacion_y_poda`, empieza con una solución inicial que reparte las tomas en orden, 6 por día, y cuyo costo es 38 con esta matriz, de modo que `mejor_coste` parte de 38. La lista de nodos empieza con la solución vacía. En cada iteración saca el último nodo añadido (así se baja en profundidad hasta soluciones completas y `mejor_coste` baja pronto) y crea sus hijos. Para cada hijo hace una de dos cosas:

- si tiene las 30 tomas, calcula su costo con `funcion_objetivo` y, si es menor que `mejor_coste`, pasa a ser la mejor solución;
- si tiene menos, calcula su `cota_inferior` y solo lo guarda si la cota es menor que `mejor_coste`. Si no lo es (por ejemplo, un nodo con cota 27 cuando `mejor_coste` ya vale 27), se poda, y con él todas las planificaciones que salían de ese nodo.

Los hijos que se guardan se añaden ordenados de mayor a menor cota para que el siguiente en salir sea el de menor cota. Cuando la lista queda vacía, `mejor_coste` es el óptimo.

In [5]:
TOMAS_POR_DIA = 6

# Devuelve el coste de la solución
def funcion_objetivo(solucion, matriz):
    n_dias = len(matriz) // TOMAS_POR_DIA
    coste = 0

    for dia in range(1, n_dias + 1):
        actores = set()

        for toma in range(len(solucion)):
            if solucion[toma] == dia:  # Se validan las tomas que están en el día
                for actor in range(len(matriz[0])):
                    if matriz[toma][actor] == 1:  # Se validan los actores que están en la toma
                        actores.add(actor)

        coste += len(actores)

    return coste

In [6]:
# Cota inferior optimista del costo de cualquier solución completa que parta de la solución parcial dada
def cota_inferior(solucion, matriz):
    n_tomas = len(matriz)
    n_dias = n_tomas // TOMAS_POR_DIA
    tomas_dia = [solucion.count(dia) for dia in range(n_dias + 1)]
    cota = 0

    for actor in range(len(matriz[0])):
        # Días a los que el actor ya va
        dias_actor = {solucion[toma] for toma in range(len(solucion)) if matriz[toma][actor] == 1}

        #Tomas del actor que todavía no tienen día
        restantes = sum(matriz[toma][actor] for toma in range(len(solucion), n_tomas))

        #Huecos libres en los días a los que ya va o nuevos
        huecos = sum(TOMAS_POR_DIA - tomas_dia[dia] for dia in dias_actor)
        nuevos = math.ceil(max(0, restantes - huecos) / TOMAS_POR_DIA)

        cota += len(dias_actor) + nuevos

    return cota

In [7]:
# ramas del árbol
def crear_hijos(solucion, n_dias):
    hijos = []

    for dia in range(1, min(max(solucion, default=0) + 1, n_dias) + 1):
        if solucion.count(dia) < TOMAS_POR_DIA:
            hijos.append({'s': solucion + (dia,)})

    return hijos

In [8]:
def ramificacion_y_poda(matriz):
    n_tomas = len(matriz)
    n_dias = n_tomas // TOMAS_POR_DIA

    # Solución inicial
    mejor_solucion = tuple(dia for dia in range(1, n_dias + 1) for _ in range(TOMAS_POR_DIA))
    mejor_coste = funcion_objetivo(mejor_solucion, matriz)

    nodos = [{'s': (), 'ci': cota_inferior((), matriz)}]

    iteracion = 0
    inicio = time.time()

    # Mientras haya nodos por explorar, aplico ramificación y poda
    while nodos:
        nodo = nodos.pop()         # Cojo el último nodo añadido
        if nodo['ci'] >= mejor_coste: # Si ya hay una solución mejor antes de añadir el nodo
            continue

        hijos_validos = []

        for hijo in crear_hijos(nodo['s'], n_dias):
            if len(hijo['s']) == n_tomas:    # Si la solución tiene todas las tomas, es una solución final
                coste = funcion_objetivo(hijo['s'], matriz)
                if coste < mejor_coste:
                    mejor_solucion = hijo['s']
                    mejor_coste = coste

            else:         # Si no, aplico poda con la cota inferior
                hijo['ci'] = cota_inferior(hijo['s'], matriz)
                # Si la cota inferior no baja del mejor coste, no puede mejorar mi solución -> Poda
                if hijo['ci'] < mejor_coste: # Si no bajo la solución entonces podo
                    hijos_validos.append(hijo)

        hijos_validos.sort(key=lambda x: x['ci'], reverse=True)            # Se añaden de mayor a menor cota
        nodos.extend(hijos_validos)

        
        # Condicional creado por si toma más de 5 horas en correr el código
        if time.time() - inicio >= 18000: 
            print(f'Después de haber corrido durante 5 horas, la mejor solución encontrada tuvo un coste de {mejor_coste} en {iteracion-1} iteraciones\n')
            return iteracion, list(mejor_solucion) ,mejor_coste # Por si se pasa más de 7minutos

        iteracion += 1

    return iteracion, list(mejor_solucion), mejor_coste

Ejecutemos con la matriz del problema y calculemos el tiempo

In [9]:
inicio = time.time()
iteraciones_rp, mejor_solucion_rp, mejor_coste_rp = ramificacion_y_poda(matriz)

print(f"Obtenida mejor solución con coste {mejor_coste_rp} en {iteraciones_rp} iteraciones ({time.time() - inicio:.0f} segundos)")

for dia in range(1, 6):
    tomas = []

    for i in range(len(mejor_solucion_rp)):
        if mejor_solucion_rp[i] == dia:
            tomas.append(i + 1)

    print("Día", dia, ":", tomas)

Obtenida mejor solución con coste 27 en 1718260 iteraciones (434 segundos)
Día 1 : [1, 10, 12, 22, 26, 29]
Día 2 : [2, 3, 4, 11, 15, 20]
Día 3 : [5, 8, 16, 21, 25, 30]
Día 4 : [6, 7, 9, 13, 27, 28]
Día 5 : [14, 17, 18, 19, 23, 24]


### Recocido Simulado

Aún habiéndolo hecho con un método óptimo me interesa trabajar con una metaheurística para comparar ambos resultados. Utilizaré entonces Recocido Simulado tomando como base el algoritmo de la Actividad Guiada 3 y haciendo los cambios necesarios para resolver este problema.

In [10]:
import random
import math
import numpy as np

# Crea una solución inicial aleatoria
def crear_solucion():
    solucion = [1]*6 + [2]*6 + [3]*6 + [4]*6 + [5]*6 # Solución más básica
    random.shuffle(solucion)
    return solucion


# Cambia de día a 2 tomas para ver cómo cambia el coste
def genera_vecina(solucion):
    vecina = solucion.copy()

    i, j = random.sample(range(30), 2)

    while solucion[i] == solucion[j]:  # Intercambio los días para 2 tomas en concreto
      i, j = random.sample(range(30), 2)

    vecina[i], vecina[j] = vecina[j], vecina[i]

    return vecina


# Parte de la temperatura
def probabilidad(delta, T):
    return math.exp(-delta/T)


def bajar_temperatura(T):
    alpha = 0.95
    return alpha*T

In [11]:
def recocido_simulado(matriz, T=1000, T_min=0.001, L=100):

    actual = crear_solucion()
    costo_actual = funcion_objetivo(actual,matriz)

    mejor = actual.copy()
    mejor_costo = costo_actual

    iteraciones= 0

    while T > T_min:

      for _ in range(L):
        vecina = genera_vecina(actual)
        costo_vecina = funcion_objetivo(vecina,matriz)

        delta = costo_vecina - costo_actual

        # Se analiza si la vecina es mejor
        if delta < 0:
          actual = vecina
          costo_actual = costo_vecina

        # Si el vecino es peor se cambia según una probabilidad que depende de T y delta
        elif random.random() < probabilidad(delta,T):
          actual = vecina
          costo_actual = costo_vecina

        # Verifico si los cambios fueron beneficiosos
        if costo_actual < mejor_costo:
          mejor = actual.copy()
          mejor_costo = costo_actual

      T = bajar_temperatura(T)
      iteraciones += 1
    return mejor, mejor_costo, iteraciones

inicio = time.time()
mejor_solucion_rs, mejor_coste_rs, iteraciones_rs = recocido_simulado(matriz)

print(f"Obtenida mejor solución con coste {mejor_coste_rs} en {iteraciones_rs} iteraciones ({time.time() - inicio:.0f} segundos)")

for dia in range(1, 6):
    tomas = []

    for i in range(len(mejor_solucion_rs)):
        if mejor_solucion_rs[i] == dia:
            tomas.append(i + 1)

    print("Día", dia, ":", tomas)

Obtenida mejor solución con coste 27 en 270 iteraciones (1 segundos)
Día 1 : [1, 5, 6, 7, 11, 26]
Día 2 : [9, 10, 12, 16, 22, 25]
Día 3 : [14, 17, 18, 19, 23, 24]
Día 4 : [2, 13, 20, 27, 28, 30]
Día 5 : [3, 4, 8, 15, 21, 29]


Con ramificación y poda se obtuvo la mejor solución con coste 27 en 1,718,260 iteraciones (434 segundos), mientras que con recocido simulado se obtuvo también un coste de 27 en 270 iteraciones (1 segundo).

Comparando ambos, los dos algoritmos llegan al mismo valor, pero el recocido simulado lo hace mucho más rápido: tarda 1 segundo frente a los 630 de la ramificación y poda. Sin embargo, el recocido simulado es una heurística y, si solo se trabajara con él, no se podría asegurar que 27 sea el mínimo global, ya que su resultado depende del azar y podría haberse quedado en un óptimo local. La ramificación y poda sí garantiza el óptimo, porque explora todo el árbol de soluciones descartando únicamente las ramas que no pueden mejorar el mejor costo. Por eso fue necesario usarla: confirma que 27 es el valor mínimo y demuestra que la solución del recocido simulado era realmente la óptima.

En conclusión, el recocido simulado es una muy buena opción cuando se busca una solución muy buena de forma rápida, y la ramificación y poda es la que permite comprobar que esa solución es la óptima, aunque con un tiempo de cálculo mucho mayor.

## 7.

(*)Calcula la complejidad del algoritmo

### Ramificacion y Poda

Para el algoritmo de ramificación y poda se usa la misma notación del punto 5: $n$ tomas, $a$ actores, $k$ tomas como máximo por día y $n/k$ días. Analicemos cada función:

- `funcion_objetivo`: recorre los $n/k$ días, para cada uno revisa las $n$ tomas y, de las que se graban ese día, recorre los $a$ actores. $O\left(n^2 \cdot \frac{a}{k}\right)$. En este algoritmo solo se llama cuando una solución está completa, es decir, una vez por cada hoja del árbol.
  
- `cota_inferior`: calcula `solucion.count(dia)` para cada día, lo que cuesta $O\left(\frac{n}{k} \cdot n\right)=O\left(\frac{n^2}{k}\right)$. Después, para cada uno de los $a$ actores, construye el conjunto de sus días y suma sus tomas restantes, ambos recorridos de $n$ tomas, y suma los huecos de $n/k$ días. En total $O\left(a \cdot n + \frac{n^2}{k}\right)$, que para el problema son $30 \cdot 10 + 30^2/6 = 450$ operaciones.
  
- `crear_hijos`: busca el mayor día usado, y para cada uno de los $n/k$ días posibles, cuenta sus tomas y construye la nueva tupla. El costo es $O\left(\frac{n^2}{k}\right)$, unas 150 operaciones.
  
- `ramificacion_y_poda`: en cada iteración saca un nodo de la lista, llama a `crear_hijos` y calcula la cota de cada uno de sus hijos, que son a lo sumo $n/k$. Ordenar y añadir esos hijos cuesta mucho menos que calcular las cotas, así que el costo de una iteración es

$$C_{iter}=O\left(\frac{n}{k} \cdot \left(a \cdot n + \frac{n^2}{k}\right)\right)=O\left(n^2 \cdot \frac{a}{k} + \frac{n^3}{k^2}\right)$$

que para el problema son $5 \cdot 450 = 2250$ operaciones por iteración. El nodo que completa la solución cuesta una llamada a `funcion_objetivo` ($O\left(n^2 \cdot \frac{a}{k}\right)$), que ya está incluida en este orden.

Falta saber cuántas iteraciones se hacen, que depende de cuánto pode la cota. En el peor caso no se poda nada y se recorre todo el árbol. Como los días son intercambiables, las hojas son las planificaciones distintas, $\frac{n!}{(k!)^{n/k} \cdot (n/k)!} \approx 1.14 \times 10^{16}$ (el cálculo del punto 1), y como cada nodo parcial se puede completar de al menos una forma y a una misma profundidad no comparten soluciones, en cada uno de los $n$ niveles hay a lo sumo ese número de nodos. Entonces el número de iteraciones es $O\left(n \cdot \frac{n!}{(k!)^{n/k} \cdot (n/k)!}\right)$ y el costo total en el peor caso es

$$T_{RP}(n,a,k)=O\left(n \cdot \frac{n!}{(k!)^{n/k} \cdot (n/k)!} \cdot \left(n^2 \cdot \frac{a}{k} + \frac{n^3}{k^2}\right)\right)$$

Con los números del problema son $30 \cdot 1.14 \times 10^{16} \cdot 2250 \approx 7.7 \times 10^{20}$ operaciones, que a $10^7$ operaciones por segundo serían unos 2.4 millones de años. Es decir, en el peor caso la ramificación y poda sigue siendo factorial.

En la práctica la poda reduce muchísimo el número de nodos. Con la matriz del problema se hicieron 1,718,260 iteraciones, alrededor de $5 \times 10^{-12}$ del árbol completo, lo que da unos $1,718,260 \cdot 2250 \approx 3.9 \times 10^{9}$ operaciones, o sea, unos 390 segundos con la misma estimación de $10^7$ operaciones por segundo (en la ejecución real tardó 630 segundos, del mismo orden). Aun así, el número de iteraciones depende de los datos y de lo ajustada que sea la cota, y no se puede acotar mejor que con el peor caso.


### Recocido Simulado

Analicemos cada función:

- `crear_solucion`: no depende del tamaño del problema, por lo que es de orden constante:
$$O(1)$$

- `funcion_objetivo`: contiene 3 bucles anidados: uno recorre los días, otro las tomas ($n$) y el último los actores ($a$). Como el número de días se puede calcular como la cantidad de tomas sobre el máximo de tomas por día ($k$), el orden es:
$$O\left(n^2 \cdot \frac{a}{k}\right)$$

- `genera_vecina`: solo intercambia 2 posiciones, por lo que es de orden constante:
$$O(1)$$

Por último, la función principal `recocido_simulado` tiene 2 bucles anidados: el primero recorre los distintos valores de temperatura, y el segundo ejecuta la cantidad de iteraciones deseadas para cada valor de temperatura. Ambos bucles tienen un número de iteraciones constante que no depende ni de la cantidad de tomas ni de la cantidad de actores. A esta combinación de parámetros la llamaremos $t$, que representa el número total de iteraciones del algoritmo, determinado por $T$, $T_{min}$, $L$ y $\alpha$.

De esta forma, la complejidad general del algoritmo queda expresada como:
$$T(n,a,k,t) = t \cdot \frac{n}{k} \cdot n \cdot a$$

Para este problema en concreto, los valores son $t=27,000$, $n=30$, $a=10$ y $k=6$, lo que da un orden aproximado de:
$$T = 27,000 \cdot \frac{30}{6} \cdot 30 \cdot 10   = 40,500,000 \text{ operaciones básicas}$$

## 8.

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

In [13]:
np.random.seed(6313738)  
matriz_prueba = np.random.randint(0,2, size = [30,10])
matriz_prueba

array([[1, 1, 0, 1, 0, 0, 0, 1, 1, 0],
       [1, 1, 1, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 0, 1],
       [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
       [1, 1, 0, 0, 1, 0, 0, 0, 0, 1],
       [1, 1, 0, 1, 1, 0, 0, 0, 0, 1],
       [0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
       [0, 1, 1, 1, 0, 1, 1, 1, 0, 0],
       [1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
       [1, 0, 1, 0, 1, 0, 1, 1, 0, 1],
       [0, 1, 0, 0, 0, 1, 0, 1, 1, 0],
       [1, 0, 1, 0, 1, 1, 1, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 1],
       [0, 1, 0, 1, 0, 1, 1, 0, 0, 1],
       [0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 0, 1, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 1, 0, 1, 1, 1],
       [1, 1, 1, 1, 0, 1, 1, 1, 0, 0],
       [0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 1, 0, 1, 1, 1, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 1, 0,

## 9.

Aplica el algoritmo al juego de datos generado

Respuesta

In [14]:
print("RAMIFICACION Y PODA")
inicio = time.time()
iteraciones_rp, mejor_solucion_rp, mejor_coste_rp = ramificacion_y_poda(matriz_prueba)
print(f"Ramificacion y Poda: coste {mejor_coste_rp} en {iteraciones_rp} iteraciones ({time.time() - inicio:.0f} segundos)")


print("\nRECOCIDO SIMULADO")
inicio = time.time()
mejor_solucion_rs, mejor_coste_rs, iteraciones_rs = recocido_simulado(matriz_prueba)
print(f"Recocido Simulado: coste {mejor_coste_rs} en {iteraciones_rs} iteraciones ({time.time() - inicio:.0f} segundos)")

RAMIFICACION Y PODA
Después de haber corrido durante 5 horas, la mejor solución encontrada tuvo un coste de 37 en 23516659 iteraciones

Ramificacion y Poda: coste 37 en 23516660 iteraciones (18000 segundos)

RECOCIDO SIMULADO
Recocido Simulado: coste 38 en 270 iteraciones (4 segundos)


Como se puede observar, aun después de 5 horas, el código de ramificación y poda no logró terminar para este grupo de datos aleatorios. Por otro lado, el recocido simulado terminó en menos de 5 segundos y obtuvo un costo de 38, solo 1 valor por encima del mejor costo encontrado por ramificación y poda, que fue de 37. Esto muestra la ventaja de utilizar una metaheurística en este tipo de problemas, ya que, aunque no garantiza encontrar el mínimo global, puede obtener soluciones muy buenas en un tiempo mucho menor.

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

1. Se hizo uso de los notebooks de las actividades guiadas para contruir tanto el código de ramificacion y poda y recocido simulado:
> Actividad Guiada 2: https://colab.research.google.com/drive/1wbegpcwV2tqOVzwv64N4w27E1PFu8L-3
> 
> Actividad Guiada 3: https://colab.research.google.com/drive/1mg6z0OknVLasxrsXLFYl8vYAQASYH9Py

2. Se hizo uso de Asistente Virtual CLAUDE para ayudarme en la revisión del cálculo de la complejidad del algoritmo únicamente para la ramificación y poda

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

- Mejorar la ramificación y poda. Con algunos juegos de datos de prueba más densos el algoritmo no terminó ni en 5 horas, y por eso se añadió el límite de tiempo. La causa principal es la cota inferior: calcula cada actor por separado, así que no tiene en cuenta que, cuando las tomas comparten muchos actores, los días se llenan de los mismos actores y el costo real es bastante mayor que la cota. Una cota que tuviera en cuenta ese solapamiento entre tomas podaría mucho antes.
- Acelerar el cálculo de la `cota_inferior` con números binarios. Ahora, en cada nodo la función vuelve a recorrer toda la solución parcial: cuenta las tomas de cada día con `count`, construye un conjunto de días por actor y suma las tomas restantes de cada uno. Se puede guardar, para cada día, qué actores hay como un único número binario, donde el bit $j$ vale 1 si el actor $j$ participa ese día (cada toma sería también un número binario, y añadirla a un día es una sola operación `|`). Con eso, saber a qué días ya va un actor se reduce a comprobar un bit, y solo hay que actualizar el día que cambia en lugar de recalcular todo. El número de tomas por día y las tomas restantes de cada actor desde cada posición también se pueden guardar y calcular una sola vez. La cota es la misma, así que el resultado no cambia y solo se reduce el tiempo por nodo. En una prueba con la matriz del problema, junto con asignar primero las tomas con más actores, el tiempo pasó de unos 170 segundos a unos 15. Aun así, esto no cambia la cota, así que con datos más densos el árbol sigue siendo demasiado grande.
- Combinar los dos métodos. Usar el resultado del recocido simulado como `mejor_coste` inicial en lugar de repartir las tomas en orden, para podar desde el primer nodo. Y a la inversa, usar la ramificación y poda en problemas pequeños para comprobar la calidad del recocido, como se hizo con 27.
- Variaciones del problema. Dejar libre el número de días (sumando un costo por día de estudio), pagar tarifas distintas a cada actor (minimizando el costo económico y no el número de actores), tener en cuenta que algunos actores no están disponibles ciertos días, que las tomas tengan distinta duración y el límite sea de horas por día en vez de 6 tomas, o que algunas tomas deban grabarse antes que otras o el mismo día.